In [343]:
%matplotlib inline

import matplotlib.pyplot as plt

import cv2
import numpy as np

from tqdm import trange
import tqdm
import random

import scipy.io

from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from kmodes.kmodes import KModes  

import math

SEED = 95667  
random.seed(SEED)
np.random.seed(SEED)

## Image display

In [344]:
def show_image_and_keypoints( image , kps ) :
    cv2.drawKeypoints( image, kps, image, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS )

    plt.figure(figsize = (10,10))
    plt.imshow(image, aspect='auto')
    plt.axis('off')
    plt.title('Keypoints and descriptors.')
    plt.show()

In [345]:
def show_top_images_grid(dataset_path, indices, id_test, ids, labels, cols=4):
    """
    Fetches the query image and its top matches, displaying them in a grid.
    cols: Number of columns you want in the grid.
    """
    images_to_show = []
    titles = []
    
    # 1. Fetch the Query Image
    label = (ids[id_test] - 1) // 80
    name = f"{dataset_path}/jpg/{label}/image_{str(ids[id_test]).zfill(4)}.jpg"
    
    image = cv2.imread(name)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    images_to_show.append(image)
    titles.append(f"Query Label: {labels[id_test]}\n(ID: {ids[id_test]})")

    indice = 0
    # 2. Fetch the Match Images from indices
    for i in indices[0]:
        label_i = labels[i]
        name = f"{dataset_path}/jpg/{label_i}/image_{str(ids[i]).zfill(4)}.jpg"
        
        image = cv2.imread(name)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        images_to_show.append(image)
        titles.append(f"{indice}° Match: ID {ids[i]} \n Label: {label_i} - {'Correct' if label_i == labels[id_test] else 'Incorrect'}")
        indice += 1 
    
    # Create the Figure and Subplots Header
    # Adjust figsize based on your layout preference (width, height)
    fig_header, axes_header = plt.subplots(1, 2, figsize=(20, 4))

    axes_header[0].imshow(images_to_show[0])
    axes_header[0].set_title(f'Query. \n Label: {labels[id_test]}\n(ID: {ids[id_test]})')
    axes_header[0].axis('off') # Hide the axis ticks and borders

    axes_header[1].imshow(images_to_show[1])
    axes_header[1].set_title(f'Sanity Test. \n Label: {labels[indices[0][0]]}\n(ID: {ids[indices[0][0]]})')
    axes_header[1].axis('off') # Hide the axis ticks and borders
    
    
    # 3. Calculate Grid Dimensions
    n_images = len(images_to_show) - 2
    rows = math.ceil(n_images / cols)
    
    # 4. Create the Figure and Subplots
    # Adjust figsize based on your layout preference (width, height)
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
    
    # Flatten axes array to easily iterate over it, even if it's a 2D grid
    if n_images > 1:
        axes = axes.flatten()
    else:
        axes = [axes] # Handle edge case where there's only 1 image
        
    # 5. Populate the Grid
    for idx in range(len(axes)):
        if idx < n_images:
            axes[idx].imshow(images_to_show[idx+2])
            axes[idx].set_title(titles[idx+2])
            axes[idx].axis('off') # Hide the axis ticks and borders
        else:
            # Turn off empty subplots if the grid isn't perfectly filled
            axes[idx].axis('off') 
            
    plt.tight_layout()
    plt.show()

## Vocabulary Creation Steps

detectar e descrever os keypoints de todas as imagens <br><br>
a junção de todos os descritores cria o vocabulario

In [346]:
def random_detect_and_describe_keypoints(image):
    kps_random = []

    img_height = image.shape[0]
    img_width = image.shape[1]

    for i in range(300):
        x = random.randint(0, img_width - 1)
        y = random.randint(0, img_height - 1)
        kps_random.append(cv2.KeyPoint(x, y, 10))
    
    return kps_random

In [347]:
def grid_detect_and_describe_keypoints(image):
    kps_grid = []

    cell_height = image.shape[0] / 20
    cell_width = image.shape[1] / 20

    for i in range(20):
        for j in range(20):
            x = int((i + 0.5) * cell_width)
            y = int((j + 0.5) * cell_height)
            kps_grid.append(cv2.KeyPoint(x, y, 10))

    return kps_grid

In [348]:
# usa algum algoritmo para detectar e descrever keypoints
def detect_and_describe_keypoints ( image, algorithm='orb' ) :
    
    image_gray = cv2.cvtColor( image , cv2.COLOR_BGR2GRAY )
        
    if algorithm in ['sift', 'random-sift', 'grid-sift']:
        keypoint = cv2.SIFT_create()
    elif algorithm in ['orb', 'orb-hamming', 'random-orb', 'grid-orb'] :
        keypoint = cv2.ORB_create()
    else :
        print('Error: algorithm not defined')
        return None
    
  
    if algorithm in ['random-orb','random-sift']:
        kps = random_detect_and_describe_keypoints(image_gray)
    elif algorithm in ['grid-orb','grid-sift']:
        kps = grid_detect_and_describe_keypoints(image_gray)
    else:
        kps = keypoint.detect(image_gray, None)

    # Describing Keypoints
    kps, descs = keypoint.compute( image_gray, kps )
    
    return kps, descs

In [349]:
def create_vocabulary ( dataset_path , algorithm='orb', show_image=False , debug=False ) :
    mat = scipy.io.loadmat( dataset_path+'/datasplits.mat' )

    ids = mat['trn1'][0] #  'val1' or 'tst1' 
    
    if algorithm == 'orb-hamming':
        train_descs = np.ndarray( shape=(0,32) , dtype=np.uint8 )
    elif algorithm in ['orb', 'random-orb', 'grid-orb']: 
        train_descs = np.ndarray( shape=(0,32) , dtype=float )
    elif algorithm in ['sift', 'random-sift', 'grid-sift']: 
        train_descs = np.ndarray( shape=(0,128) , dtype=float )
    else :
        print('Error:Algorithm not defined.')
        return None
    
    for id in tqdm.tqdm(ids, desc='Processing train set') :
    
        label = (id - 1) // 80
        name = dataset_path + '/jpg/' + str(label) + '/image_' + str(id).zfill(4) + '.jpg'

        image = cv2.imread( name )
        
        if image is None:
            print(f'Reading image Error. Path: {name}')
            return None

        kps, descs = detect_and_describe_keypoints ( image, algorithm=algorithm )
            
        if algorithm == 'orb-hamming':
            descs = descs.astype(np.uint8)  # Garante binário
        elif algorithm in ['orb', 'random-orb', 'grid-orb']:
            descs = descs.astype(np.float32)  
            
        train_descs = np.concatenate((train_descs, descs), axis=0)
        
        if show_image :
            show_image_and_keypoints(image, kps)

        if debug :
            print( name )
            print( 'Number of keypoints: ', len(kps) )
            print( 'Number of images: ', len(ids) )
            print( 'Descriptor size: ', len(descs[0]) )
            print( type(descs[0]) )
      
    print( ' -> [I] Image Loader Info:\n', 
      '\nTrain len: ', len(train_descs),
      '\nNumber of images: ', len(ids),
      '\nDescriptor size: ', len(descs[0])      
      )
    
    return train_descs

## Create Dictionary

faz a clusterização dos descritores (vocabulário)
os centroides representam o dicionário

In [350]:
def create_dictionary_kmeans ( vocabulary ,     # descritores
                               num_cluster,
                               algorithm='sift') :  # numero de cluster
  
    print( ' -> [I] Dictionary Info:\n', 
        '\nTrain len: ', len(vocabulary),
        '\nDimension: ', len(vocabulary[0]), # varia de acordo com o descritor usado
        '\nClusters: ', num_cluster 
        )

    # objeto para fazer a clusterizacao
    if algorithm == 'orb-hamming':
        if vocabulary.dtype != np.uint8:
            vocabulary = vocabulary.astype(np.uint8)
        dictionary = KModes( n_clusters=num_cluster, init='Huang', n_init=1,  max_iter=50, verbose=0 )
    else:
        dictionary = MiniBatchKMeans( n_clusters=num_cluster, batch_size=1000, random_state=SEED )
    
    print ( 'Learning dictionary by Kmeans...')
    # clusteriza o vocabulary
    # retorna o dictionary treinado
    dictionary = dictionary.fit( vocabulary )

    # dictionary.cluster_centers_  sao os centroides
    print ( 'Done.')

    return dictionary

## Image Representation

cria o histograma que representa a imagem

In [351]:
def create_bovw_descriptors (image, dictionary, algorithm='orb') :
    
    descs = detect_and_describe_keypoints( image, algorithm=algorithm )[1] # retorna os keypoints e seus descritores

    if algorithm == 'orb-hamming':
        predicted = dictionary.predict(descs.astype(np.uint8))
        hist = np.histogram(predicted, bins=range(0, dictionary.n_clusters + 1))[0]
        desc_bovw = (hist > 0).astype(np.uint8)
    else: 
        if descs.dtype != np.double:
            descs = descs.astype(np.double)
        predicted = dictionary.predict(np.array(descs, dtype=np.double))
        desc_bovw = np.histogram(predicted, bins=range(0, dictionary.n_clusters+1))[0]

    return desc_bovw

## Describe Dataset Images

cria um descritor para cada imagem do dataset

In [352]:
def represent_dataset( dataset_path, dictionary, test=False, algorithm='orb' ) :

    mat = scipy.io.loadmat( dataset_path+'/datasplits.mat' )

    if test :
        ids = mat['tst1'][0] #  'tst1' or 'trn1' or 'val1'
    else :
        ids = mat['val1'][0] #  'tst1' or 'trn1' or 'val1'
    
    space = []
    labels = []
    
    for id in tqdm.tqdm(ids, desc='Processing test set' if test else 'Processing val set') :

        label = (id - 1) // 80
        name = dataset_path + '/jpg/' + str(label) + '/image_' + str(id).zfill(4) + '.jpg'

        image = cv2.imread( name )
        
        if image is None:
            print(f'Reading image Error. Path: {name}')
            return None

        desc_bovw = create_bovw_descriptors(image, dictionary, algorithm=algorithm)

        space.append(desc_bovw)
        labels.append(label)
        
    print( ' -> [I] Space Describing Info:\n', 
        '\nNumber of images: ', len(space), 
        '\nNumber of labels: ', len(labels),
        '\nDimension: ', len(space[0])
        )

    return space , labels 

## Acurracy

mede a precisão do image retrieval

In [353]:
def run_experiment ( space , labels , dictionary , dataset_path, test=False, algorithm='orb', top=10 ) :
    if algorithm == 'orb-hamming':
        knn = NearestNeighbors(n_neighbors=top+1, metric='hamming').fit(space)
    else: 
        knn = NearestNeighbors(n_neighbors=top+1, metric='euclidean').fit(space)
    
    mat = scipy.io.loadmat( dataset_path+'/datasplits.mat' )

    if test :
        ids = mat['tst1'][0] #  'tst1' or 'trn1' or 'val1'
    else :
        ids = mat['val1'][0] #  'tst1' or 'trn1' or 'val1'
    
    accuracy_t = 0
    
    for id_test in tqdm.tqdm(ids, desc= 'running the test phase' if test else 'running the val phase') :
        
        label = (id_test - 1) // 80
        name = dataset_path + '/jpg/' + str(label) + '/image_' + str(id_test).zfill(4) + '.jpg'

        image = cv2.imread( name )
        
        desc_bovw = create_bovw_descriptors(image, dictionary, algorithm=algorithm)

        indices = knn.kneighbors(desc_bovw.reshape(1, -1))[1] #retorna as imagens mais parecidas    

        labels_top = [ labels[i] for i in indices[0] ]

        accuracy = sum( np.equal(labels_top, label) )
        accuracy =( (accuracy-1)/(top) ) * 100
        accuracy_t = accuracy_t + accuracy

    avg_acc = accuracy_t / len(ids)

    print(f'Average accuracy in the', 'test' if test else 'validation', f'set: {avg_acc:5.2f}%')

    return avg_acc 

## Image Retrieval

as k imagens mais proximas

In [354]:
def retrieve_single_image ( space , labels , dictionary , dataset_path, algorithm='orb', top=10 ) :
    if algorithm == 'orb-hamming':
        knn = NearestNeighbors(n_neighbors=top+1, metric='hamming').fit(space)
    else: 
        knn = NearestNeighbors(n_neighbors=top+1, metric='euclidean').fit(space)
    
    
    mat = scipy.io.loadmat( dataset_path+'/datasplits.mat' )

    ids = mat['tst1'][0] #  'trn1' or 'val1'
    
    id_test = random.randrange( len(ids) )
        
    label = (ids[id_test] - 1) // 80
    name = dataset_path + '/jpg/' + str(label) + '/image_' + str(ids[id_test]).zfill(4) + '.jpg'
    
    image = cv2.imread( name )
    
    if image is None:
        print(f'Reading image Error. Path: {name}')
        return None

    desc_bovw = create_bovw_descriptors(image, dictionary, algorithm=algorithm)
    
    distances, indices = knn.kneighbors(desc_bovw.reshape(1, -1))
    
    # show_top_images(dataset_path, indices, id_test, ids, labels)
    show_top_images_grid(dataset_path, indices, id_test, ids, labels)
    
    labels_top = [ int(labels[i]) for i in indices[0] ]
    
    accuracy = sum( np.equal( label , labels_top ) )
    accuracy =( (accuracy-1)/(top) ) * 100 
    
    print(f'Accuracy for image id {ids[id_test]}: {accuracy:5.2f}%')
    
    print(name)    
    print(f'Image: {ids[id_test]} with label {labels[id_test]}')    
    print(f'Closest image: {ids[indices[0][0]]} with distance {distances[0][0]} and label {labels[indices[0][0]]}')
    print('Distances: ',distances)
    print('Indices: ',indices[0])
    print('Labels: ',labels_top)

## Results

In [355]:
import pandas as pd

def show_results_table(results):
    # ============================================
    # TABELA 1: Algoritmos BoVW (com Dic. Size)
    # ============================================
    
    # Filtra apenas algoritmos que NÃO são LBP
    bovw_results = {k: v for k, v in results.items() if k != 'LBP'}
    
    if bovw_results:
        df_bovw = pd.DataFrame(bovw_results)
        df_bovw.index.name = "Dic. Size"
        df_bovw = df_bovw.reset_index()
        
        # Converte para inteiro
        df_bovw["Dic. Size"] = df_bovw["Dic. Size"].astype(int)
        
        # Arredonda valores
        df_bovw = df_bovw.round(2)
        
        # Função para destacar o melhor da linha
        def highlight_max(row):
            numeric_cols = row[1:]
            max_val = numeric_cols.max()
            return ['' if col == "Dic. Size" else
                    'background-color: green' if row[col] == max_val else ''
                    for col in row.index]
        
        print("\n" + "="*70)
        print("TABELA 1: BoVW Algorithms (different dictionary sizes)")
        print("="*70)
        display(df_bovw.style.apply(highlight_max, axis=1))
        
        # Encontra melhor configuração global (excluindo LBP)
        best = None
        best_val = 0
        
        for alg in bovw_results:
            for k in bovw_results[alg]:
                if bovw_results[alg][k] > best_val:
                    best_val = bovw_results[alg][k]
                    best = (alg, k)
        
        if best:
            print(f"\n🏆 Best BoVW configuration: Algorithm = {best[0].upper()}, K = {best[1]} → {best_val:.2f}%")
    
    # ============================================
    # TABELA 2: LBP (sem Dic. Size)
    # ============================================
    
    if 'LBP' in results and results['LBP']:
        print("\n" + "="*70)
        print("TABELA 2: LBP (global descriptor)")
        print("="*70)
        
        # Cria DataFrame com 1 linha e 1 coluna
        lbp_acc = results['LBP'].get('N/A', list(results['LBP'].values())[0] if results['LBP'] else 0)
        
        df_lbp = pd.DataFrame({
            'Algorithm': ['LBP'],
            'Accuracy (%)': [round(lbp_acc, 2)]
        })
        
        display(df_lbp)
        print(f"\n📊 LBP Average Accuracy: {lbp_acc:.2f}%")

## LBP

In [356]:
from skimage.feature import local_binary_pattern

def extract_lbp(image_gray, radius=1, n_points=8):
    """Extrai histograma LBP da imagem"""
    lbp = local_binary_pattern(image_gray, n_points, radius, method='uniform')
    
    n_bins = n_points + 2
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins))
    
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-7)
    
    return hist

def represent_dataset_lbp(dataset_path, test=False):
    """Representa dataset usando LBP (não BoVW)"""
    mat = scipy.io.loadmat(dataset_path+'/datasplits.mat')
    
    if test:
        ids = mat['tst1'][0]
    else:
        ids = mat['val1'][0]
    
    space = []
    labels = []
    
    for id in tqdm.tqdm(ids, desc='Processing LBP'):
        label = (id - 1) // 80
        name = f"{dataset_path}/jpg/{label}/image_{str(id).zfill(4)}.jpg"
        
        image = cv2.imread(name)
        if image is None:
            print(f'Error: {name}')
            return None
        
        image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        lbp_hist = extract_lbp(image_gray)
        
        space.append(lbp_hist)
        labels.append(label)
    
    return space, labels

def run_experiment_lbp(space, labels, dataset_path, test=False, top=10):
    """Executa experimento com LBP usando distância Euclidiana"""
    knn = NearestNeighbors(n_neighbors=top+1, metric='euclidean').fit(space)
    
    mat = scipy.io.loadmat(dataset_path+'/datasplits.mat')
    
    if test:
        ids = mat['tst1'][0]
    else:
        ids = mat['val1'][0]
    
    accuracy_t = 0
    
    for id_test in tqdm.tqdm(ids, desc='running LBP'):
        label = (id_test - 1) // 80
        name = f"{dataset_path}/jpg/{label}/image_{str(id_test).zfill(4)}.jpg"
        
        image = cv2.imread(name)
        image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        lbp_hist = extract_lbp(image_gray)
        
        indices = knn.kneighbors(lbp_hist.reshape(1, -1))[1]
        labels_top = [labels[i] for i in indices[0]]
        
        accuracy = sum(np.equal(labels_top, label))
        accuracy = ((accuracy-1)/(top)) * 100
        accuracy_t += accuracy
    
    avg_acc = accuracy_t / len(ids)
    return avg_acc

## Tune

In [357]:
algorithms = [ "random-sift", "random-orb", "grid-sift", "grid-orb", "orb", "orb-hamming", "sift"]
clusters = [100, 200, 400, 600, 800]
dataset_path = '/Users/nicholasbarbosa/Mestrado/inf692/flowers_classes'

results = {alg: {} for alg in algorithms}
results["LBP"] = {}  # LBP não usa clusters


# Rodar LBP separadamente
print("\n" + "="*50)
print("ALGORITHM: LBP")
print("="*50)

space_lbp, labels_lbp = represent_dataset_lbp(dataset_path, test=False)
acc_lbp = run_experiment_lbp(space_lbp, labels_lbp, dataset_path, test=False)
results["LBP"]["N/A"] = acc_lbp
print(f"Average accuracy (LBP): {acc_lbp:.2f}%")

for algorithm in algorithms:
    print("\n" + "="*50)
    print(f"ALGORITHM: {algorithm.upper()}")
    print("="*50)
    
    print("\nStep 1: Vocabulary Creation")
    print("-"*50)
    vocabulary = create_vocabulary(dataset_path, algorithm=algorithm)

    for num_cluster in clusters:
        print("\n" + "-"*50)
        print(f"Step 2: Dictionary Creation | K = {num_cluster}")
        print("-"*50)
        
        dictionary = create_dictionary_kmeans(vocabulary, num_cluster=num_cluster, algorithm=algorithm)
        
        print("\nStep 3: Represent Dataset (BoVW)")
        print("-"*50)
        space, labels = represent_dataset(dataset_path, dictionary, algorithm=algorithm)

        print("\nStep 4: Running Experiment (Validation)")
        print("-"*50)
        acc = run_experiment(space, labels, dictionary, dataset_path, algorithm=algorithm)

        # salva resultado
        results[algorithm][num_cluster] = acc

show_results_table(results)


ALGORITHM: LBP


running LBP: 100%|██████████| 340/340 [00:05<00:00, 67.35it/s]


Average accuracy (LBP): 7.85%

ALGORITHM: RANDOM-SIFT

Step 1: Vocabulary Creation
--------------------------------------------------


Processing train set: 100%|██████████| 680/680 [00:35<00:00, 19.06it/s]


 -> [I] Image Loader Info:
 
Train len:  204000 
Number of images:  680 
Descriptor size:  128

--------------------------------------------------
Step 2: Dictionary Creation | K = 100
--------------------------------------------------
 -> [I] Dictionary Info:
 
Train len:  204000 
Dimension:  128 
Clusters:  100
Learning dictionary by Kmeans...
Done.

Step 3: Represent Dataset (BoVW)
--------------------------------------------------


...

Average accuracy in the validation set: 22.09%

TABELA 1: BoVW Algorithms (different dictionary sizes)


,Dic. Size,random-sift,random-orb,grid-sift,grid-orb,orb,orb-hamming,sift
0,100,14.350000,2.530000,15.530000,9.410000,15.760000,6.740000,19.970000
1,200,16.090000,3.210000,17.210000,8.620000,15.560000,7.850000,21.380000
2,400,15.290000,2.560000,16.410000,8.790000,15.350000,11.790000,22.060000
3,600,16.940000,2.210000,16.680000,8.260000,14.560000,13.680000,21.940000
4,800,16.240000,2.090000,18.240000,9.470000,14.150000,13.090000,22.090000



🏆 Best BoVW configuration: Algorithm = SIFT, K = 800 → 22.09%

TABELA 2: LBP (global descriptor)


,Algorithm,Accuracy (%)
0,LBP,7.85



📊 LBP Average Accuracy: 7.85%
